# Actividad 3 | Aprendizaje supervisado y no supervisado
---
- César Iván Pedrero Martínez   |   A01366501

## 1. Construcción de la muestra M
---------------------------------------------
En esta sección se generarán las particiones Mi de la muestra M a partir del dataset original (P),
usando criterios de particionamiento basados en las variables: star_rating, verified_purchase, y vine.
Además, se realizará un muestreo estratificado por sentimiento para mantener la distribución original de clases.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import findspark
from os import path

In [2]:
findspark.init()
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/07 19:06:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
PATH = "../files"
FILE = "amazon_electronics.csv"

In [4]:
RELEVANT_COLUMNS_FOR_CHARACTERIZATION = [
    "star_rating", "helpful_votes", "total_votes",
    "vine", "verified_purchase", "review_date", "sentiment"
]

In [ ]:
class PartitioningManager:
  @staticmethod
  def compute_probabilities(df, cols):
    total_count = df.count()
    return df.groupBy(cols).count() \
            .withColumn("probability", F.round(F.col("count") / total_count, 6)) \
            .orderBy("probability", ascending=False)

  @staticmethod
  def filter_partition(df, star_rating, verified_purchase, vine):
    return df.filter(
      (col("star_rating") == star_rating) &
      (col("verified_purchase") == verified_purchase) &
      (col("vine") == vine)
    )

  @staticmethod
  def generate_all_partitions(df, min_probability=0.0001, min_rows=100):
    prob_df = PartitioningManager.compute_probabilities(
      df, ["star_rating", "verified_purchase", "vine"]
    )

    filtered_combinations = prob_df.filter(
      F.col("probability") >= min_probability
    ).select("star_rating", "verified_purchase", "vine").collect()

    partitions = {}
    for row in filtered_combinations:
      rating, purchase, vine_status = row["star_rating"], row["verified_purchase"], row["vine"]
      key = f"R{rating}_VP{purchase}_V{vine_status}"

      filtered = PartitioningManager.filter_partition(df, rating, purchase, vine_status)
      count = filtered.count()

      if count >= min_rows:
        partitions[key] = filtered
        print(f"✔️  Partición {key} creada con {count} registros.")
      else:
        print(f"⚠️  Partición {key} descartada por bajo volumen ({count} registros).")

    return partitions

  @staticmethod
  def stratified_sample_partitioned_data(partitions_dict, label_col="sentiment", fraction=0.05, min_rows=100):
    sampled_partitions = {}

    for key, df in partitions_dict.items():
      count = df.count()
      if count < min_rows:
        print(f"❌ Saltando partición {key} (< {min_rows} registros)")
        continue

      sentiments = df.select(label_col).distinct().rdd.flatMap(lambda x: x).collect()
      fractions = {s: fraction for s in sentiments}

      sampled_df = df.sampleBy(label_col, fractions, seed=42)
      sampled_partitions[key] = sampled_df
      print(f"🧪 Partición {key}: {sampled_df.count()} registros muestreados de {count}.")

    return sampled_partitions

  @staticmethod
  def build_combined_sample(partitions_sampled_dict):
    combined_df = None
    for key, df in partitions_sampled_dict.items():
      combined_df = df if combined_df is None else combined_df.union(df)
      print(f"🔗 Añadida partición {key} a muestra M.")

    print(f"✅ Muestra M construida con {combined_df.count()} registros totales.")
    return combined_df

In [6]:
class FileManager():
  @staticmethod
  def open_csv_file(input_path: str, file_name: str):
    """
    Abre el archivo CSV y devuelve un DataFrame de PySpark.
    """
    return spark.read.csv(
      path.join(input_path, file_name),
      header=True,
      inferSchema=True,
      multiLine=True,
      escape="\"",
      quote="\""
    )

In [7]:
df_reviews = FileManager.open_csv_file(PATH, FILE)
df_reviews

marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date,sentiment
US,22873041,R3ARRMDEGED8RD,B00KJWQIIC,335625766,Plemo 14-Inch Lap...,PC,5,0,0,N,Y,Pleasantly surprised,I was very surpri...,2015-08-31,1
US,30088427,RQ28TSA020Y6J,B013ALA9LA,671157305,TP-Link OnHub AC1...,PC,5,24,31,N,N,OnHub is a pretty...,I am a Google emp...,2015-08-31,1
US,20329786,RUXJRZCT6953M,B00PML2GQ8,982036237,AmazonBasics USB ...,PC,1,2,2,N,N,None of them work...,Bought cables in ...,2015-08-31,0
US,14215710,R7EO0UO6BPB71,B001NS0OZ4,576587596,Transcend P8 15-i...,PC,1,0,0,N,Y,just keep searching.,"nope, cheap and slow",2015-08-31,0
US,38264512,R39NJY2YJ1JFSV,B00AQMTND2,964759214,Aleratec SATA Dat...,PC,5,0,0,N,Y,Five Stars,Excellent! Great ...,2015-08-31,1
US,30548466,R31SR7REWNX7CF,B00KX4TORI,170101802,Kingston Digital ...,PC,5,0,0,N,Y,"Good quality, wor...","Good quality,work...",2015-08-31,1
US,589298,RVBP8I1R0CTZ8,B00P17WEMY,206124740,White 9 Inch Unlo...,PC,3,1,2,N,Y,in fact this is t...,This demn tablet ...,2015-08-31,0
US,49329488,R1QF6RS1PDLU18,B00TR05L9Y,778403103,Lenovo TAB2 A10 -...,PC,4,1,1,N,Y,Good,I am not sure I d...,2015-08-31,1
US,50728290,R23AICGEDAJQL1,B0098Y77OG,177098042,Acer,PC,1,0,0,N,Y,You get what you ...,After exactly 45 ...,2015-08-31,0
US,37802374,R2EY3N4K9W19UP,B00IFYEYXC,602496520,AzureWave Broadco...,PC,5,3,4,N,Y,Great for Windows...,Replaced my Intel...,2015-08-31,1


In [8]:
df_reviews_filtered = df_reviews.select(*RELEVANT_COLUMNS_FOR_CHARACTERIZATION)
df_reviews_filtered

star_rating,helpful_votes,total_votes,vine,verified_purchase,review_date,sentiment
5,0,0,N,Y,2015-08-31,1
5,24,31,N,N,2015-08-31,1
1,2,2,N,N,2015-08-31,0
1,0,0,N,Y,2015-08-31,0
5,0,0,N,Y,2015-08-31,1
5,0,0,N,Y,2015-08-31,1
3,1,2,N,Y,2015-08-31,0
4,1,1,N,Y,2015-08-31,1
1,0,0,N,Y,2015-08-31,0
5,3,4,N,Y,2015-08-31,1


Vamos a generar las particiones usando las estrategias de las actividades pasadas. Lo haremos en 3 pasos:
- Paso 1: Generar particiones válidas (con mínimo volumen y probabilidad significativa).
- Paso 2: Aplicar muestreo estratificado por sentimiento en cada Mi.
- Paso 3: Unir todas las particiones Mi para construir la muestra M.

In [ ]:
partitions = PartitioningManager.generate_all_partitions(
    df_reviews_filtered,
    min_probability=0.00005,
    min_rows=100
)

sampled_partitions = PartitioningManager.stratified_sample_partitioned_data(
    partitions,
    fraction=0.05,
    min_rows=100
)

df_sample_M = PartitioningManager.build_combined_sample(sampled_partitions)

df_sample_M.show(5)
df_sample_M.groupBy("sentiment").count().show()

25/06/07 19:06:31 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


✔️  Partición R5_VPY_VN creada con 3679909 registros.


✔️  Partición R4_VPY_VN creada con 1019728 registros.


✔️  Partición R1_VPY_VN creada con 603371 registros.


✔️  Partición R3_VPY_VN creada con 443364 registros.


...

🧪 Partición R3_VPN_VY: 266 registros muestreados de 4886.


🧪 Partición R2_VPN_VY: 93 registros muestreados de 1634.


🧪 Partición R1_VPN_VY: 35 registros muestreados de 705.
🔗 Añadida partición R5_VPY_VN a muestra M.
🔗 Añadida partición R4_VPY_VN a muestra M.
🔗 Añadida partición R1_VPY_VN a muestra M.
🔗 Añadida partición R3_VPY_VN a muestra M.
🔗 Añadida partición R5_VPN_VN a muestra M.
🔗 Añadida partición R2_VPY_VN a muestra M.
🔗 Añadida partición R1_VPN_VN a muestra M.
🔗 Añadida partición R4_VPN_VN a muestra M.
🔗 Añadida partición R3_VPN_VN a muestra M.
🔗 Añadida partición R2_VPN_VN a muestra M.
🔗 Añadida partición R5_VPN_VY a muestra M.
🔗 Añadida partición R4_VPN_VY a muestra M.
🔗 Añadida partición R3_VPN_VY a muestra M.
🔗 Añadida partición R2_VPN_VY a muestra M.
🔗 Añadida partición R1_VPN_VY a muestra M.


✅ Muestra M construida con 345826 registros totales.
+-----------+-------------+-----------+----+-----------------+-----------+---------+
|star_rating|helpful_votes|total_votes|vine|verified_purchase|review_date|sentiment|
+-----------+-------------+-----------+----+-----------------+-----------+---------+
|          5|            0|          0|   N|                Y| 2015-08-31|        1|
|          5|            0|          0|   N|                Y| 2015-08-31|        1|
|          5|            0|          0|   N|                Y| 2015-08-31|        1|
|          5|            0|          0|   N|                Y| 2015-08-31|        1|
|          5|            0|          0|   N|                Y| 2015-08-31|        1|
+-----------+-------------+-----------+----+-----------------+-----------+---------+
only showing top 5 rows



+---------+------+
|sentiment| count|
+---------+------+
|        1|263986|
|        0| 81840|
+---------+------+



## 2. Construcción Train – Test

En esta sección se construyen los conjuntos de entrenamiento (Train) y prueba (Test) a partir de la muestra M generada anteriormente. Para evitar sesgos, se realiza una división estratificada de cada partición Mi que compone M, asegurando que:
- Se mantenga la distribución de clases (sentiment) dentro de cada subconjunto.
- No exista intersección entre Train y Test.
- La unión de todos los Tri conforma el conjunto de entrenamiento global, y la unión de todos los Tsi conforma el conjunto de prueba global.


In [ ]:
class TrainTestBuilder:
  @staticmethod
  def split_stratified_partition(df, label_col="sentiment", test_ratio=0.2):
    """
    Divide un DataFrame en entrenamiento y prueba de manera estratificada
    según la columna de clase (label_col).
    """
    unique_labels = df.select(label_col).distinct().rdd.flatMap(lambda x: x).collect()
    fractions_test = {label: test_ratio for label in unique_labels}

    df_test = df.sampleBy(label_col, fractions_test, seed=123)
    df_train = df.subtract(df_test)

    return df_train, df_test

  @staticmethod
  def split_all_partitions(sampled_partitions, label_col="sentiment", test_ratio=0.2):
    """
    Aplica división estratificada a todas las particiones Mi y retorna
    conjuntos globales de entrenamiento y prueba.
    """
    train_partitions = []
    test_partitions = []

    for key, partition_df in sampled_partitions.items():
      df_train, df_test = TrainTestBuilder.split_stratified_partition(
        partition_df,
        label_col=label_col,
        test_ratio=test_ratio
      )
      train_partitions.append(df_train)
      test_partitions.append(df_test)
      print(f"🔀 {key} → Train: {df_train.count()} | Test: {df_test.count()}")

    df_train_global = train_partitions[0]
    df_test_global = test_partitions[0]

    for df in train_partitions[1:]:
      df_train_global = df_train_global.union(df)

    for df in test_partitions[1:]:
      df_test_global = df_test_global.union(df)

    print(f"\n✅ Conjunto global de entrenamiento: {df_train_global.count()} registros")
    print(f"✅ Conjunto global de prueba: {df_test_global.count()} registros")

    return df_train_global, df_test_global

Solo queda ejecutar el proceso sobre las particiones Mi muestreadas previamente.

In [ ]:
df_train, df_test = TrainTestBuilder.split_all_partitions(
  sampled_partitions,
  label_col="sentiment",
  test_ratio=0.2
)

🔀 R5_VPY_VN → Train: 13788 | Test: 36848


🔀 R4_VPY_VN → Train: 7730 | Test: 10344


🔀 R1_VPY_VN → Train: 9688 | Test: 6146


🔀 R3_VPY_VN → Train: 5618 | Test: 4530


🔀 R5_VPN_VN → Train: 7576 | Test: 4204


...

🔀 R3_VPN_VY → Train: 204 | Test: 54


🔀 R2_VPN_VY → Train: 77 | Test: 16


🔀 R1_VPN_VY → Train: 31 | Test: 4



✅ Conjunto global de entrenamiento: 63832 registros


✅ Conjunto global de prueba: 69738 registros


Veamos una visualización básica para validación

In [ ]:
df_train.groupBy("sentiment").count().show()
df_test.groupBy("sentiment").count().show()

+---------+-----+
|sentiment|count|
+---------+-----+
|        1|33823|
|        0|30009|
+---------+-----+



+---------+-----+
|sentiment|count|
+---------+-----+
|        1|53091|
|        0|16647|
+---------+-----+



## 3. Selección de métricas para medir calidad de resultados

Para evaluar objetivamente el desempeño de los modelos de clasificación entrenados, es fundamental definir métricas que consideren tanto la precisión general del modelo como su comportamiento ante clases desbalanceadas. En este proyecto, se trabaja con una tarea de clasificación binaria (sentiment: 1 = positivo, 0 = negativo), por lo que las métricas seleccionadas deberán:

1. Ser apropiadas para tareas de clasificación.
2. Ser eficientes computacionalmente ante grandes volúmenes de datos.
3. Proporcionar información detallada sobre los tipos de errores cometidos por el modelo.

### Métricas seleccionadas

**1. Accuracy (Precisión global):**  
Proporción de predicciones correctas sobre el total de instancias. Aunque es útil como referencia general, puede ser engañosa si las clases están desbalanceadas.

**2. Precision (Precisión por clase positiva):**  
Fracción de instancias predichas como positivas que realmente lo son. Es útil cuando el costo de un falso positivo es alto.

**3. Recall (Exhaustividad por clase positiva):**  
Fracción de instancias positivas que fueron correctamente identificadas. Es importante cuando el costo de un falso negativo es elevado.

**4. F1-Score:**  
Media armónica entre precision y recall. Se utiliza para balancear ambos criterios, especialmente cuando hay desbalance de clases.

**5. Área bajo la curva ROC (AUC-ROC):**  
Mide la capacidad del modelo para discriminar entre clases. Es robusta ante desbalance y útil para comparar modelos.

Estas métricas permiten una evaluación completa del desempeño del modelo, tanto en términos generales (accuracy y AUC) como específicos (precision, recall y F1). Además, están soportadas nativamente por bibliotecas como `pyspark.ml.evaluation` y pueden ser calculadas de manera eficiente sobre conjuntos grandes.

Estas métricas serán implementadas en la etapa de experimentación del paso 4.


In [ ]:
class MetricsEvaluator:
  @staticmethod
  def evaluate_binary_classification(predictions: DataFrame, label_col: str = "label", prediction_col: str = "prediction", probability_col: str = "probability"):
    """
    Evalúa un modelo de clasificación binaria en base a varias métricas estándar.
    """
    accuracy = predictions.filter(F.col(label_col) == F.col(prediction_col)).count() / predictions.count()
    
    tp = predictions.filter((F.col(prediction_col) == 1) & (F.col(label_col) == 1)).count()
    fp = predictions.filter((F.col(prediction_col) == 1) & (F.col(label_col) == 0)).count()
    fn = predictions.filter((F.col(prediction_col) == 0) & (F.col(label_col) == 1)).count()
    
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0.0

    evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol=probability_col, metricName="areaUnderROC")
    auc = evaluator.evaluate(predictions)

    print(f"📊 Métricas del modelo:")
    print(f"   - Accuracy:  {accuracy:.4f}")
    print(f"   - Precision: {precision:.4f}")
    print(f"   - Recall:    {recall:.4f}")
    print(f"   - F1 Score:  {f1:.4f}")
    print(f"   - AUC-ROC:   {auc:.4f}")

    return {
      "accuracy": accuracy,
      "precision": precision,
      "recall": recall,
      "f1_score": f1,
      "auc": auc
    }

## 4. Entrenamiento de modelos de aprendizaje

### Algoritmo seleccionado

Para esta etapa se seleccionó **Logistic Regression**, un modelo de aprendizaje supervisado ampliamente utilizado para clasificación binaria. Es adecuado para grandes volúmenes de datos y proporciona salidas probabilísticas interpretables.

### Estrategia de entrenamiento

Se utiliza el conjunto de entrenamiento `df_train` derivado en la etapa 2. El flujo de trabajo incluye:

1. Preprocesamiento de variables de entrada.
2. Ensamblado de variables predictoras.
3. Entrenamiento del modelo.
4. Ajuste de hiperparámetros mediante validación cruzada.
5. Evaluación con el conjunto de prueba `df_test`.
6. Prevención de sobreajuste con regularización L2 y validación cruzada.

Las métricas de evaluación fueron definidas en la sección anterior.


In [ ]:
class ModelTrainer:
    @staticmethod
    def preprocess_data(df_train, df_test, feature_cols, label_col="sentiment"):
        """
        Arma un pipeline para ensamblar y escalar características.
        """
        assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_vec")
        scaler = StandardScaler(inputCol="features_vec", outputCol="features", withStd=True, withMean=False)
        pipeline = Pipeline(stages=[assembler, scaler])

        pipeline_model = pipeline.fit(df_train)
        train_preprocessed = pipeline_model.transform(df_train)
        test_preprocessed = pipeline_model.transform(df_test)

        return train_preprocessed, test_preprocessed

    @staticmethod
    def train_logistic_regression(df_train, df_test, label_col="sentiment"):
        """
        Entrena un modelo de regresión logística con validación cruzada y evaluación.
        """
        lr = LogisticRegression(labelCol=label_col, featuresCol="features", maxIter=10)

        paramGrid = (ParamGridBuilder()
            .addGrid(lr.regParam, [0.1])
            .addGrid(lr.elasticNetParam, [0.0])
            .build())

        evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC")

        cv = CrossValidator(estimator=lr,
                            estimatorParamMaps=paramGrid,
                            evaluator=evaluator,
                            numFolds=2,
                            parallelism=2)

        cv_model = cv.fit(df_train)
        best_model = cv_model.bestModel

        predictions = best_model.transform(df_test)

        print(f"✅ Mejor modelo entrenado: RegParam = {best_model._java_obj.getRegParam()}, ElasticNet = {best_model._java_obj.getElasticNetParam()}")
        
        return predictions, best_model

Hagamos el pre-procesamiento en algunas variables relevantes.

In [ ]:
features = ["star_rating", "helpful_votes", "total_votes"]

train_pp, test_pp = ModelTrainer.preprocess_data(
  df_train.sample(withReplacement=False, fraction=0.1, seed=42),
  df_test.sample(withReplacement=False, fraction=0.1, seed=42),
  feature_cols=features
)

Empecemos el entrenamiento!

In [ ]:
predictions, best_model = ModelTrainer.train_logistic_regression(train_pp, test_pp)

25/06/07 19:25:20 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/06/07 19:25:20 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


✅ Mejor modelo entrenado: RegParam = 0.1, ElasticNet = 0.0


Finalmente podemos evaluar las métricas definidas en la sección 3.

In [ ]:
metrics = MetricsEvaluator.evaluate_binary_classification(predictions, label_col="sentiment")

📊 Métricas del modelo:
   - Accuracy:  0.8741
   - Precision: 0.8612
   - Recall:    0.8427
   - F1 Score:  0.8518
   - AUC-ROC:   0.9024


## 5. Análisis de resultados

Los resultados obtenidos tras el entrenamiento del modelo de regresión logística permiten realizar un análisis crítico tanto del rendimiento general como de sus implicancias prácticas en una tarea de clasificación binaria, en este caso, análisis de sentimiento.

#### Fortalezas

* **Buen rendimiento general:** El modelo alcanzó una **precisión de 86.12%**, un **recall de 84.27%**, y un **F1 Score de 85.18%**, lo cual indica un buen equilibrio entre falsos positivos y falsos negativos. Estos valores sugieren que el modelo es capaz de capturar correctamente una gran proporción de los ejemplos positivos, sin perder mucha precisión en el proceso.

* **Área bajo la curva (AUC-ROC):** Con un valor de **0.9024**, el modelo muestra una gran capacidad para diferenciar entre las clases positivas y negativas. Esto es especialmente relevante en contextos donde el costo de errores puede ser alto, ya que un AUC elevado sugiere que el modelo asigna puntuaciones adecuadas a las instancias.

* **Consistencia con validación cruzada:** El uso de validación cruzada ayudó a evitar el sobreajuste, lo cual se refleja en métricas razonables sobre el conjunto de prueba. Esto indica una generalización aceptable hacia datos no vistos.

#### Áreas de oportunidad

* **Posible sesgo en el conjunto de datos:** A pesar de los buenos resultados, el rendimiento del modelo podría estar influido por la distribución del conjunto de datos. Si una clase está sobrerrepresentada (por ejemplo, más opiniones positivas que negativas), las métricas pueden parecer infladas. Un análisis más detallado de la distribución de clases y un posible reequilibrado (por ejemplo, mediante submuestreo o SMOTE) podrían mejorar aún más la robustez.

* **Limitaciones del modelo lineal:** Aunque la regresión logística es interpretable y eficiente, su capacidad de capturar relaciones no lineales entre características es limitada. Explorar modelos más complejos como árboles de decisión, Random Forest o Gradient Boosting podría revelar patrones más sutiles en los datos y mejorar las métricas.

* **Escalabilidad y rendimiento en producción:** Aunque el modelo es eficiente, se debe evaluar su rendimiento en contextos de datos en tiempo real o con volúmenes mucho mayores. A su vez, la latencia y la estabilidad del pipeline completo (preprocesamiento + predicción) deben ser testeadas si se plantea su despliegue.

En resumen, el modelo entrenado presenta un desempeño adecuado para tareas de análisis de sentimiento, con métricas sólidas y una buena capacidad de generalización. Sin embargo, existen oportunidades claras para mejorar la representación de los datos y explorar modelos más sofisticados.